In [8]:
#import necessary package
import torch
import copy
import torch.nn.utils.prune as prune
from torchvision import transforms, datasets, models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [9]:
# preprocess images
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # MobileNetV2 default input size
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])  # ImageNet normalization
])

train_dataset = datasets.ImageFolder("../data/dataset/Training", transform=transform)
test_dataset   = datasets.ImageFolder("../data/dataset/Test", transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader   = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)
# Load model
model = torch.load("fruit_mobilenetv2.pth")

# Training SetUp
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [10]:
# evaluate function
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

# calculate sensitivity of each layer
def measure_layer_sensitivity(model, layer_name, test_loader, device):
    temp_model = copy.deepcopy(model)
    temp_model.to(device)
    temp_model.eval()

    # find the layer
    module = dict(temp_model.named_modules())[layer_name]

    # prune 10% of filters
    prune.ln_structured(module, name="weight", amount=0.1, n=1, dim=0)
    prune.remove(module, "weight")

    # evaluate accuracy
    acc = evaluate(temp_model,test_loader, device)

    return acc


In [11]:
sensitivities = {}
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        print(f"Estimate sensitivity of layer: {name}")
        acc = measure_layer_sensitivity(model, name, test_loader, device)
        print(f"Validation Accuracy: {acc:.2f}%")
        sensitivities[name] = acc

Estimate sensitivity of layer: features.0.0
Validation Accuracy: 89.35%
Estimate sensitivity of layer: features.1.conv.0.0
Validation Accuracy: 89.35%
Estimate sensitivity of layer: features.1.conv.1
Validation Accuracy: 42.97%
Estimate sensitivity of layer: features.2.conv.0.0
Validation Accuracy: 87.17%
Estimate sensitivity of layer: features.2.conv.1.0
Validation Accuracy: 88.05%
Estimate sensitivity of layer: features.2.conv.2
Validation Accuracy: 71.56%
Estimate sensitivity of layer: features.3.conv.0.0
Validation Accuracy: 87.42%
Estimate sensitivity of layer: features.3.conv.1.0
Validation Accuracy: 87.21%
Estimate sensitivity of layer: features.3.conv.2
Validation Accuracy: 78.88%
Estimate sensitivity of layer: features.4.conv.0.0
Validation Accuracy: 89.37%
Estimate sensitivity of layer: features.4.conv.1.0
Validation Accuracy: 65.81%
Estimate sensitivity of layer: features.4.conv.2
Validation Accuracy: 81.73%
Estimate sensitivity of layer: features.5.conv.0.0
Validation Accur

In [12]:
# Use structured pruning: Assign pruning ratio based on sensitivity
originalModel_acc = evaluate(model, test_loader, device)

pruning_plan = {}

for layer, acc in sensitivities.items():
    drop = originalModel_acc - acc

    if drop < 0.5:
        pruning_plan[layer] = 0.5   # prune 50%
    elif drop < 1.0:
        pruning_plan[layer] = 0.3   # prune 30%
    elif drop < 2.0:
        pruning_plan[layer] = 0.1   # prune 10%
    else:
        pruning_plan[layer] = 0.0   # too sensitive, skip

# Apply pruning plan into model
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        amount = pruning_plan[name]
        if amount > 0:
            prune.ln_structured(module, name="weight", amount=amount, n=1, dim=0)
            prune.remove(module, "weight")
            print(f"Pruned {amount*100:.0f}% of filters in {name}")


Pruned 50% of filters in features.0.0
Pruned 50% of filters in features.1.conv.0.0
Pruned 10% of filters in features.2.conv.1.0
Pruned 10% of filters in features.3.conv.0.0
Pruned 50% of filters in features.4.conv.0.0
Pruned 50% of filters in features.5.conv.0.0
Pruned 30% of filters in features.5.conv.2
Pruned 50% of filters in features.6.conv.0.0
Pruned 50% of filters in features.6.conv.1.0
Pruned 30% of filters in features.6.conv.2
Pruned 50% of filters in features.7.conv.0.0
Pruned 10% of filters in features.7.conv.2
Pruned 50% of filters in features.8.conv.0.0
Pruned 50% of filters in features.8.conv.2
Pruned 50% of filters in features.9.conv.0.0
Pruned 10% of filters in features.9.conv.1.0
Pruned 10% of filters in features.10.conv.0.0
Pruned 10% of filters in features.10.conv.1.0
Pruned 10% of filters in features.10.conv.2
Pruned 50% of filters in features.11.conv.0.0
Pruned 50% of filters in features.11.conv.2
Pruned 30% of filters in features.12.conv.0.0
Pruned 50% of filters i

In [13]:
# fine-tune the pruned model
for epoch in range(5):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    acc = evaluate(model, test_loader, device)
    print(f"Epoch {epoch+1}, Test Accuracy: {acc:.2f}%")





Epoch 1, Test Accuracy: 79.70%
Epoch 2, Test Accuracy: 79.54%
Epoch 3, Test Accuracy: 78.60%
Epoch 4, Test Accuracy: 79.02%
Epoch 5, Test Accuracy: 78.76%


In [14]:
torch.save(model, "../model/prunned_fruit_mobilenetv2.pth")

NameError: name 'sensitivities' is not defined